In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [2]:
def encodeLabel(data, feature):
    encoder = LabelEncoder()
    data[feature] = encoder.fit_transform(data[feature].fillna('none'))
    return data

In [3]:
driver = pd.read_csv('../data/driver.csv')

In [4]:
item = pd.read_csv('../data/data/item_data.csv')
item = encodeLabel(item, 'brand')
item = encodeLabel(item, 'brand_type')
item = encodeLabel(item, 'category')

In [5]:
data = pd.read_csv('../data/data/customer_transaction_data.csv')
data = data[data['coupon_discount'] == 0]
data = data.merge(item[['item_id','brand']], on='item_id').drop('item_id', axis=1)
data = data.groupby(['customer_id','brand'])[['selling_price','quantity']].sum().reset_index()

In [6]:
mapping = pd.read_csv('../data/data/coupon_item_mapping.csv')
mapping = mapping.merge(item, on='item_id')[['coupon_id','brand']]
mapping = mapping.drop_duplicates()

In [7]:
data = data.merge(mapping, on=['brand'])

In [8]:
data.head()

,customer_id,brand,selling_price,quantity,coupon_id
0,1,0,697.44,24,24
1,1,0,697.44,24,33
2,1,0,697.44,24,7
3,1,0,697.44,24,20
4,1,0,697.44,24,29


In [9]:
feat_1 = data.groupby(['customer_id','coupon_id'])['quantity'].sum().reset_index()
feat_1 = feat_1.rename(columns={'quantity':'sum_trx_cust_brnd_qty'})
feat_2 = data.groupby(['customer_id','coupon_id'])['selling_price'].sum().reset_index()
feat_2 = feat_2.rename(columns={'selling_price':'sum_trx_cust_brnd_price'})

In [10]:
driver = driver.merge(feat_1, on=['customer_id','coupon_id'], how='left')
driver = driver.merge(feat_2, on=['customer_id','coupon_id'], how='left')
driver = driver.fillna(0)

In [11]:
driver = driver.drop(['campaign_id','coupon_id','customer_id'], axis=1)

In [12]:
driver.to_csv('../data/feature/tranx_brand_feature.csv', index=False)

In [13]:
driver.shape

(128595, 3)

In [14]:
driver.head()

,id,sum_trx_cust_brnd_qty,sum_trx_cust_brnd_price
0,1,0.0,0.00
1,2,11708.0,9843.52
2,6,0.0,0.00
3,7,0.0,0.00
4,9,0.0,0.00
